### The environment 🎮

- [LunarLander-v3](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

### The library used 📚

- [Stable-Baselines3](https://stable-baselines3.readthedocs.io/en/master/)

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the Github Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives of this notebook 🏆

At the end of the notebook, you will:

- Be able to use **Gymnasium**, the environment library.
- Be able to use **Stable-Baselines3**, the deep reinforcement learning library.
- Be able to **push your trained agent to the Hub** with a nice video replay and an evaluation score 🔥.
- Train **SAC** with **dict** observations (`state` + image `pixels`) using **MultiInputPolicy**.




In this free course, you will:

- 📖 Study Deep Reinforcement Learning in **theory and practice**.
- 🧑‍💻 Learn to **use famous Deep RL libraries** such as Stable Baselines3, RL Baselines3 Zoo, CleanRL and Sample Factory 2.0.
- 🤖 Train **agents in unique environments**
- 🎓 **Earn a certificate of completion** by completing 80% of the assignments.

And more!

Check 📚 the syllabus 👉 https://simoninithomas.github.io/deep-rl-course

Don’t forget to **<a href="http://eepurl.com/ic5ZUD">sign up to the course</a>** (we are collecting your email to be able to **send you the links when each Unit is published and give you information about the challenges and updates).**

The best way to keep in touch and ask questions is **to join our discord server** to exchange with the community and with us 👉🏻 https://discord.gg/ydHrjt3WP5

## Prerequisites 🏗️

Before diving into the notebook, you need to:

🔲 📝 **[Read Unit 0](https://huggingface.co/deep-rl-course/unit0/introduction)** that gives you all the **information about the course and helps you to onboard** 🤗

🔲 📚 **Develop an understanding of the foundations of Reinforcement learning** (RL process, Rewards hypothesis...) by [reading Unit 1](https://huggingface.co/deep-rl-course/unit1/introduction).

Let's do a small recap on what we learned in the first Unit:

- Reinforcement Learning is a **computational approach to learning from actions**. We build an agent that learns from the environment by **interacting with it through trial and error** and receiving rewards (negative or positive) as feedback.

- The goal of any RL agent is to **maximize its expected cumulative reward** (also called expected return) because RL is based on the _reward hypothesis_, which is that all goals can be described as the maximization of an expected cumulative reward.

- The RL process is a **loop that outputs a sequence of state, action, reward, and next state**.

- To calculate the expected cumulative reward (expected return), **we discount the rewards**: the rewards that come sooner (at the beginning of the game) are more probable to happen since they are more predictable than the long-term future reward.

- To solve an RL problem, you want to **find an optimal policy**; the policy is the "brain" of your AI that will tell us what action to take given a state. The optimal one is the one that gives you the actions that max the expected return.

There are **two** ways to find your optimal policy:

- By **training your policy directly**: policy-based methods.
- By **training a value function** that tells us the expected return the agent will get at each state and use this function to define our policy: value-based methods.

- Finally, we spoke about Deep RL because **we introduce deep neural networks to estimate the action to take (policy-based) or to estimate the value of a state (value-based) hence the name "deep."**

# Let's train our first Deep Reinforcement Learning agent and upload it to the Hub 🚀

## Get a certificate 🎓

To validate this hands-on for the [certification process](https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process), you need to push your trained model to the Hub and **get a result of >= 200**.

To find your result, go to the [leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) and find your model, **the result = mean_reward - std of reward**

For more information about the certification process, check this section 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## Install dependencies and virtual screen (outside this notebook)

All **pip** packages are listed in [`requirements.txt`](requirements.txt). Run the install commands in a terminal or in Colab **before** running the rest of this notebook — **the cells below do not run any installs**.

Keep next to the notebook: `requirements.txt`, `lunar_rl_common.py`, and optionally `best_hyperparams.json` if you tune hyperparameters outside this notebook.

**System (Linux / Colab)** — tools for Box2D, rendering, and Xvfb:

```bash
sudo apt-get update
sudo apt-get install -y swig cmake python3-opengl ffmpeg xvfb
```

**Python**:

```bash
pip install -r requirements.txt
```

On **Google Colab**: upload the files via the file browser, then run the two command blocks above (for example in a one-off code cell you delete after installing).


Replay videos need a **virtual display** (Xvfb). Make sure you installed the **xvfb** system package from the section above.

If your environment recommends a restart after installs, run the next restart cell, then the virtual-display cell.


Sometimes, for newly installed libraries to load correctly, you need to **restart the notebook runtime**. The next cell forces a **deliberate process crash** — reconnect, then run again starting from the virtual-display cell below.

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## Import the packages 📦

One additional library we import is huggingface_hub **to be able to upload and download trained models from the hub**.


The Hugging Face Hub 🤗 works as a central place where anyone can share and explore models and datasets. It has versioning, metrics, visualizations and other features that will allow you to easily collaborate with others.

You can see here all the Deep reinforcement Learning models available here👉 https://huggingface.co/models?pipeline_tag=reinforcement-learning&sort=downloads



In [ ]:
# Reproducibility: fixed seed for reproducible runs
import random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
import torch
import torch.nn as nn
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

import gymnasium

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login  # To log to our Hugging Face account to be able to upload models to the Hub.

from stable_baselines3 import SAC
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor


At each step:
- Our Agent receives a **state (S0)** from the **Environment** — we receive the first frame of our game (Environment).
- Based on that **state (S0),** the Agent takes an **action (A0)** — our Agent will move to the right.
- The environment transitions to a **new** **state (S1)** — new frame.
- The environment gives some **reward (R1)** to the Agent — we’re not dead *(Positive Reward +1)*.


With Gymnasium:

1️⃣ We create our environment using `gymnasium.make()`

2️⃣ We reset the environment to its initial state with `observation = env.reset()`

At each step:

3️⃣ Get an action using our model (in our example we take a random action)

4️⃣ Using `env.step(action)`, we perform this action in the environment and get
- `observation`: The new state (st+1)
- `reward`: The reward we get after executing the action
- `terminated`: Indicates if the episode terminated (agent reach the terminal state)
- `truncated`: Introduced with this new version, it indicates a timelimit or if an agent go out of bounds of the environment for instance.
- `info`: A dictionary that provides additional information (depends on the environment).

For more explanations check this 👉 https://gymnasium.farama.org/api/env/#gymnasium.Env.step

If the episode is terminated:
- We reset the environment to its initial state with `observation = env.reset()`

**Let's look at an example!** Make sure to read the code


In [ ]:
import gymnasium as gym

# First, we create our environment called LunarLander-v3
try:
    env = gym.make("LunarLander-v3")
except gym.error.DependencyNotInstalled:
    # Install Box2D dependencies if missing (e.g. after runtime restart)
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "swig"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gymnasium[box2d]"])
    env = gym.make("LunarLander-v3")

# Then we reset this environment (seed for reproducibility)
observation, info = env.reset(seed=SEED)
terminated, truncated = False, False
for _ in range(5):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
        observation, info = env.reset(seed=SEED)
        terminated, truncated = False, False
env.close()


## Create the LunarLander environment 🌛 and understand how it works

### [The environment 🎮](https://gymnasium.farama.org/environments/box2d/lunar_lander/)

In this first tutorial, we’re going to train our agent, a [Lunar Lander](https://gymnasium.farama.org/environments/box2d/lunar_lander/), **to land correctly on the moon**. To do that, the agent needs to learn **to adapt its speed and position (horizontal, vertical, and angular) to land correctly.**

---


💡 A good habit when you start to use an environment is to check its documentation

👉 https://gymnasium.farama.org/environments/box2d/lunar_lander/

---


Let's see what the Environment looks like:


In [ ]:
# We create our environment with gym.make("<name_of_the_environment>")
env = gym.make("LunarLander-v3")
env.reset(seed=SEED)
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample()) # Get a random observation

We see with `Observation Space Shape (8,)` that the observation is a vector of size 8, where each value contains different information about the lander:
- Horizontal pad coordinate (x)
- Vertical pad coordinate (y)
- Horizontal speed (x)
- Vertical speed (y)
- Angle
- Angular speed
- If the left leg contact point has touched the land (boolean)
- If the right leg contact point has touched the land (boolean)

**In this notebook**, SAC training uses a **dict** observation (`state`: this 8-d vector after `VecNormalize`, plus `pixels`: resized grayscale frames). The cells above use the **unwrapped** env to show the native vector space.


In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action

The action space (the set of possible actions the agent can take) is discrete with 4 actions available 🎮:

- Action 0: Do nothing,
- Action 1: Fire left orientation engine,
- Action 2: Fire the main engine,
- Action 3: Fire right orientation engine.

Reward function (the function that will give a reward at each timestep) 💰:

After every step a reward is granted. The total reward of an episode is the **sum of the rewards for all the steps within that episode**.

For each step, the reward:

- Is increased/decreased the closer/further the lander is to the landing pad.
-  Is increased/decreased the slower/faster the lander is moving.
- Is decreased the more the lander is tilted (angle not horizontal).
- Is increased by 10 points for each leg that is in contact with the ground.
- Is decreased by 0.03 points each frame a side engine is firing.
- Is decreased by 0.3 points each frame the main engine is firing.

The episode receive an **additional reward of -100 or +100 points for crashing or landing safely respectively.**

An episode is **considered a solution if it scores at least 200 points.**

#### Vectorized Environment

- We create a vectorized environment (a method for stacking multiple independent environments into a single environment) of **8** parallel environments, this way, **we'll have more diverse experiences during the training.**


In [ ]:
# ============================================================
# Hyperparameters — edit this cell to tune your training
# ============================================================

import os

# --- Environment ---
env_id = "LunarLander-v3"
n_envs = 32                     # parallel environments (SubprocVecEnv)

# --- Training ---
total_timesteps = 16_000_000   # full training run
model_name = "sac-LunarLander-v3"
checkpoint_dir = "./checkpoints"
vecnormalize_path = os.path.join(checkpoint_dir, "vecnormalize.pkl")
save_freq = 500_000            # checkpoint every N *total* env steps (approx)

# SAC defaults (overwritten if you load best_hyperparams.json below)
learning_rate_max = 3e-4       # peak LR for schedule
learning_rate_min = 1e-5       # floor LR at end of training
lr_schedule = "cosine"         # "cosine" or "linear"

buffer_size = 1_000_000
learning_starts = 10_000
batch_size = 256
tau = 0.005
gamma = 0.99
gradient_steps = 1
train_freq = 1
tensorboard_log = os.path.join(checkpoint_dir, "sac_tensorboard")

target_entropy = -2.0

# --- Periodic evaluation (EvalCallback counts VecEnv step calls) ---
periodic_eval_episodes = 10
periodic_eval_freq = max(10_000 // n_envs, 1)

# --- Progress video ---
video_eval_freq = 200_000

# --- Live reward plot ---
reward_plot_window = 50
reward_plot_freq = 5000

# --- Optional: JSON from offline hyperparameter search ---
use_best_hyperparams_json = False
best_params_path = "best_hyperparams.json"


In [ ]:
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from lunar_rl_common import (
    LUNAR_LANDER_TARGET_ENTROPY,
    NET_ARCH_PRESETS,
    EntropyLogCallback,
    PeriodicEvalCallback,
    VecNormalizeSaveCallback,
    cosine_lr_schedule,
    get_device,
    linear_lr_schedule,
    make_eval_vec_env_with_stats,
    make_lunar_dict_env,
    make_subproc_venv,
    make_train_vec_env,
    sac_policy_kwargs,
    make_sac,
)

device = get_device()


In [ ]:
from stable_baselines3.common.env_checker import check_env

_val_env = make_lunar_dict_env()
check_env(_val_env, warn=True, skip_render_check=True)
obs, _ = _val_env.reset(seed=SEED)
assert set(obs.keys()) == {"state", "pixels"}
assert obs["state"].shape == (8,) and obs["state"].dtype == np.float32
assert obs["pixels"].shape == (1, 84, 84) and obs["pixels"].dtype == np.uint8
print("Validation OK:", {k: (obs[k].shape, obs[k].dtype) for k in obs})
_val_env.close()


In [ ]:
# Smoke test: SAC + replay buffer + VecNormalize (~8k steps, then eval)
import os
import tempfile

smoke_n = 2
smoke_steps = 8_000
smoke_learning_starts = 5_000
smoke_dir = tempfile.mkdtemp(prefix="lunar_sac_smoke_")
smoke_vec = os.path.join(smoke_dir, "vecnormalize.pkl")

def _smoke_factory():
    return make_lunar_dict_env()

smoke_venv = DummyVecEnv([_smoke_factory] * smoke_n)
smoke_venv.seed(SEED)
smoke_venv.reset()
smoke_venv = VecNormalize(
    smoke_venv,
    norm_obs=True,
    norm_reward=True,
    clip_obs=10.0,
    clip_reward=10.0,
    gamma=gamma,
    norm_obs_keys=["state"],
)
obs = smoke_venv.reset()
print("Smoke obs keys:", list(obs.keys()))

smoke_lr = cosine_lr_schedule(3e-4, 1e-5)
smoke_model = make_sac(
    smoke_venv,
    policy_kwargs=sac_policy_kwargs(NET_ARCH_PRESETS["256_256"]),
    learning_rate=smoke_lr,
    buffer_size=min(50_000, buffer_size),
    learning_starts=smoke_learning_starts,
    batch_size=min(128, batch_size),
    tau=tau,
    gamma=gamma,
    train_freq=1,
    gradient_steps=1,
    ent_coef="auto",
    target_entropy=LUNAR_LANDER_TARGET_ENTROPY,
    seed=SEED,
    device=device,
    verbose=0,
)
smoke_model.learn(total_timesteps=smoke_steps)
smoke_venv.save(smoke_vec)

eval_smoke = make_eval_vec_env_with_stats(smoke_vec, SEED)
mean_r, std_r = evaluate_policy(
    smoke_model, eval_smoke, n_eval_episodes=3, deterministic=True
)
eval_smoke.close()
print(f"Smoke eval mean_reward={mean_r:.2f} std={std_r:.2f}")

smoke_venv.close()
del smoke_model
print("Smoke test finished (buffer + eval OK).")


In [ ]:
env = make_train_vec_env(n_envs=n_envs, seed=SEED, gamma=gamma)


In [ ]:
import json
import os

best_params_path = "best_hyperparams.json"
if use_best_hyperparams_json and os.path.isfile(best_params_path):
    with open(best_params_path, "r", encoding="utf-8") as f:
        best_data = json.load(f)
    best = best_data["params"]
    learning_rate_max = best["learning_rate"]
    gamma = best["gamma"]
    tau = best["tau"]
    batch_size = best["batch_size"]
    net_key = best.get("net_arch", "256_256")
    net_arch_tuple = NET_ARCH_PRESETS.get(net_key, NET_ARCH_PRESETS["256_256"])
    policy_kwargs = sac_policy_kwargs(net_arch_tuple)
    print(f"Loaded hyperparameters from {best_params_path!r}")
    print(f"  mean_reward={best_data.get('mean_reward')} std={best_data.get('std_reward')}")
else:
    net_arch_tuple = NET_ARCH_PRESETS["256_256"]
    policy_kwargs = sac_policy_kwargs(net_arch_tuple)
    if use_best_hyperparams_json:
        print(f"No file at {best_params_path!r} — using defaults.")


## Create the Model 🤖
- We have studied our environment and we understood the problem: **being able to land the Lunar Lander to the Landing Pad correctly by controlling left, right and main orientation engine**. Now let's build the algorithm we're going to use to solve this Problem 🚀.

- To do so, we're going to use our first Deep RL library, [Stable Baselines3 (SB3)](https://stable-baselines3.readthedocs.io/en/master/).

- SB3 is a set of **reliable implementations of reinforcement learning algorithms in PyTorch**.

---

💡 A good habit when using a new library is to dive first on the documentation: https://stable-baselines3.readthedocs.io/en/master/ and then try some tutorials.

----

To solve this problem, we're going to use SB3 **SAC**. [SAC (Soft Actor-Critic) is one of the SOTA (state of the art) Deep Reinforcement Learning algorithms that you'll study during this course](https://stable-baselines3.readthedocs.io/en/master/modules/sac.html#example%5D).

SAC learns a stochastic policy and Q-functions with entropy regularization of:
- *Value-based reinforcement learning method*: learning an action-value function that will tell us the **most valuable action to take given a state and action**.
- *Policy-based reinforcement learning method*: learning a policy that will **give us a probability distribution over actions**.

Stable-Baselines3 is easy to set up:

1️⃣ You **create your environment** (in our case it was done above)

2️⃣ You define the **model you want to use and instantiate this model** `model = SAC("MultiInputPolicy", ...)`

3️⃣ You **train the agent** with `model.learn` and define the number of training timesteps

```
# Create environment
env = make_train_vec_env(...)

# Instantiate the agent
model = SAC('MultiInputPolicy', env, verbose=1, policy_kwargs=policy_kwargs, buffer_size=..., ...)
# Train the agent
model.learn(total_timesteps=int(2e5))
```



In [ ]:
lr_callable = (
    cosine_lr_schedule(learning_rate_max, learning_rate_min)
    if lr_schedule.lower() == "cosine"
    else linear_lr_schedule(learning_rate_max, learning_rate_min)
)

model = make_sac(
    env,
    policy_kwargs=policy_kwargs,
    learning_rate=lr_callable,
    buffer_size=buffer_size,
    learning_starts=learning_starts,
    batch_size=batch_size,
    tau=tau,
    gamma=gamma,
    train_freq=train_freq,
    gradient_steps=gradient_steps,
    ent_coef="auto",
    target_entropy=target_entropy,
    seed=SEED,
    device=device,
    verbose=1,
    tensorboard_log=tensorboard_log,
)


## One observation as seen by the model

A **single** example (env index 0 in the batch): the same **dict** structure passed to `model.predict` / the network — `state` and `pixels` after **VecNormalize** (vector only). Also shows the image as **uint8** from the env and after SB3 **normalize_images**.

This cell builds a temporary `DummyVecEnv` and copies normalization statistics from `env` so the **training** vec env is not reset.

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# Demo env — does not reset the training env
_demo = DummyVecEnv([make_lunar_dict_env])
_demo.seed(SEED + 12345)
_demo.reset()
_demo_vn = VecNormalize(
    _demo,
    norm_obs=True,
    norm_reward=False,
    clip_obs=10.0,
    clip_reward=10.0,
    gamma=gamma,
    norm_obs_keys=["state"],
)
if isinstance(env, VecNormalize):
    _demo_vn.obs_rms = copy.deepcopy(env.obs_rms)
else:
    print("(env is not VecNormalize yet — state uses fresh running stats only)")

_obs = _demo_vn.reset()
idx = 0

print("=== Observation structure (NumPy, as input to model.predict) ===")
print("Type:", type(_obs).__name__)
for k in sorted(_obs.keys()):
    v = np.asarray(_obs[k])
    print(f"  [{k!r}] shape={v.shape}, dtype={v.dtype}")
    if k == "state":
        print(
            "       env #0 values:",
            np.array2string(v[idx], precision=4, suppress_small=True),
        )
    elif k == "pixels":
        print(f"       env #0 min/max: {int(v[idx].min())} / {int(v[idx].max())}")

tensors, _ = model.policy.obs_to_tensor(_obs)
print("\n=== After policy preprocessing (tensors for the network) ===")
for k in sorted(tensors.keys()):
    t = tensors[k]
    print(f"  [{k!r}] shape={tuple(t.shape)}, dtype={t.dtype}, device={t.device}")
    u = t[idx].detach().cpu()
    if k == "state":
        print(
            "       values:",
            np.array2string(u.numpy().squeeze(), precision=4, suppress_small=True),
        )
    elif k == "pixels":
        uu = u.squeeze(0)
        print(f"       min/max: {uu.min().item():.4f} / {uu.max().item():.4f}")

px = np.asarray(_obs["pixels"][idx]).squeeze(0)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(px, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("pixels from env (uint8, before /255)")
axes[0].axis("off")
pp = tensors["pixels"][idx].detach().cpu().squeeze(0).numpy()
axes[1].imshow(pp, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("pixels after normalize_images (float, for network)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

_demo_vn.close()

In [ ]:
# Verify GPU is used (run after creating the model)
print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(model.policy.parameters()).device)
# Env stepping (Box2D + rendering) is CPU-heavy; GPU mainly runs the CNN/MLP updates.


## Train the SAC agent 🏃
- Let's train our agent for 1,000,000 timesteps, don't forget to use GPU on Colab. It will take approximately ~20min, but you can use fewer timesteps if you just want to try it out.
- During the training, take a ☕ break you deserved it 🤗

In [ ]:
model_name = "sac-LunarLander-v3"


In [ ]:
import os
import numpy as np
from matplotlib import animation, pyplot as plt
from IPython.display import display, HTML
from stable_baselines3.common.callbacks import BaseCallback


class VideoProgressCallback(BaseCallback):
    """Record and display a deterministic-policy video every eval_freq training steps."""

    def __init__(self, eval_freq=200_000, vecnormalize_path=None, seed=42):
        super().__init__(verbose=0)
        self.eval_freq = eval_freq
        self.vecnormalize_path = vecnormalize_path
        self.seed = seed
        self._last_eval = 0

    def _on_step(self) -> bool:
        if self.num_timesteps - self._last_eval >= self.eval_freq:
            self._last_eval = self.num_timesteps
            self._record_and_show()
        return True

    def _record_and_show(self):
        if not os.path.isfile(self.vecnormalize_path):
            return
        eval_venv = make_eval_vec_env_with_stats(self.vecnormalize_path, self.seed)
        eval_venv.seed(self.seed)
        obs = eval_venv.reset()
        frames = []
        fr = eval_venv.env_method("render")[0]
        if fr is not None:
            frames.append(fr)
        total_reward = 0.0
        while True:
            action, _ = self.model.predict(obs, deterministic=True)
            obs, reward, done, info = eval_venv.step(action)
            total_reward += float(reward[0])
            fr = eval_venv.env_method("render")[0]
            if fr is not None:
                frames.append(fr)
            if done[0]:
                break
        eval_venv.close()

        if not frames:
            return
        fig, ax = plt.subplots(figsize=(6, 4), dpi=72)
        ax.axis("off")
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        im = ax.imshow(frames[0])

        def update(i):
            im.set_array(frames[i])
            return [im]

        anim = animation.FuncAnimation(
            fig, update, frames=len(frames), interval=33, blit=True
        )
        video_html = anim.to_html5_video()
        plt.close(fig)

        display(
            HTML(
                f"<h4>Step {self.num_timesteps:,} | Reward: {total_reward:.1f}</h4>"
                f"{video_html}"
            )
        )


video_cb = VideoProgressCallback(
    eval_freq=video_eval_freq,
    vecnormalize_path=vecnormalize_path,
    seed=SEED,
)


In [ ]:
import ipywidgets as widgets
import numpy as np
from stable_baselines3.common.callbacks import BaseCallback
from IPython.display import display

class LiveRewardPlotCallback(BaseCallback):
    """Live-updating episode reward plot during training."""

    def __init__(self, window=50, plot_freq=5000, verbose=0):
        super().__init__(verbose)
        self.window = window
        self.plot_freq = plot_freq
        self.episode_rewards = []
        self.episode_timesteps = []
        self.output = widgets.Output()

    def _on_training_start(self):
        display(self.output)

    def _on_step(self) -> bool:
        infos = self.locals.get("infos", [])
        for info in infos:
            ep = info.get("episode")
            if ep is not None:
                self.episode_rewards.append(ep["r"])
                self.episode_timesteps.append(self.num_timesteps)

        if (self.num_timesteps % self.plot_freq < self.training_env.num_envs
                and len(self.episode_rewards) >= self.window):
            self._update_plot()
        return True

    def _update_plot(self):
        rews = np.array(self.episode_rewards)
        ts = np.array(self.episode_timesteps)

        mean = np.convolve(rews, np.ones(self.window) / self.window, mode='valid')
        std = np.array([
            rews[max(0, i - self.window):i].std()
            for i in range(self.window, len(rews) + 1)
        ])
        score = mean - std
        ts_valid = ts[self.window - 1:]

        with self.output:
            self.output.clear_output(wait=True)
            fig, ax1 = plt.subplots(figsize=(12, 5))

            ax1.plot(ts_valid, mean, color='#2196F3', linewidth=2, label='Mean reward')
            ax1.plot(ts_valid, score, color='#FF9800', linewidth=2, label='Score (mean - std)')
            ax1.axhline(y=200, color='green', linestyle='--',
                        alpha=0.7, label='Solved (200)')
            ax1.set_xlabel('Timesteps')
            ax1.set_ylabel('Reward')
            ax1.grid(True, alpha=0.3)

            ax2 = ax1.twinx()
            ax2.plot(ts_valid, std, color='#E53935', linewidth=1.5,
                     alpha=0.7, linestyle=':', label='Std')
            ax2.set_ylabel('Std', color='#E53935')
            ax2.tick_params(axis='y', labelcolor='#E53935')

            lines1, labels1 = ax1.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right')

            latest_mean = mean[-1]
            latest_std = std[-1]
            latest_score = score[-1]
            ax1.set_title(
                f'Training Progress ({len(self.episode_rewards)} episodes) | '
                f'Mean: {latest_mean:.1f}  Std: {latest_std:.1f}  '
                f'Score: {latest_score:.1f}')
            plt.tight_layout()
            plt.show()

live_plot_cb = LiveRewardPlotCallback(window=reward_plot_window, plot_freq=reward_plot_freq)

In [ ]:
import os
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.vec_env import VecNormalize

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(tensorboard_log, exist_ok=True)

_checkpoint_every = max(save_freq // n_envs, 1)
checkpoint_callback = CheckpointCallback(
    save_freq=_checkpoint_every,
    save_path=checkpoint_dir,
    name_prefix=model_name,
)
vecnormalize_cb = VecNormalizeSaveCallback(
    save_freq=_checkpoint_every,
    save_path=vecnormalize_path,
)
periodic_eval_cb = PeriodicEvalCallback(
    stats_path=vecnormalize_path,
    seed=SEED,
    eval_freq=periodic_eval_freq,
    n_eval_episodes=periodic_eval_episodes,
    env_id=env_id,
    verbose=1,
)
entropy_log_cb = EntropyLogCallback(log_freq=max(periodic_eval_freq, 1), verbose=1)

model.learn(
    total_timesteps=total_timesteps,
    callback=[
        checkpoint_callback,
        vecnormalize_cb,
        periodic_eval_cb,
        entropy_log_cb,
        video_cb,
        live_plot_cb,
    ],
)

if isinstance(env, VecNormalize):
    env.save(vecnormalize_path)
model.save(os.path.join(checkpoint_dir, f"{model_name}-final"))


## Evaluate the agent 📈
- The training env uses **dict** observations (`state` + `pixels`) and **VecNormalize** on `state` only. Evaluation must use the **same wrapped env** and load **`vecnormalize.pkl`** (`training=False`, `norm_reward=False`) so the policy sees the same normalized inputs as during training.
- Stable-Baselines3 provides `evaluate_policy` for this.
- In the next step, we'll see **how to automatically evaluate and share your agent** on the leaderboard; here we measure performance locally.


💡 Do not evaluate on a raw vector-only `LunarLander-v3` env — the **MultiInputPolicy** expects dict observations with matching normalization statistics.

In [ ]:
eval_env = make_eval_vec_env_with_stats(vecnormalize_path, SEED)
mean_reward, std_reward = evaluate_policy(
    model, eval_env, n_eval_episodes=20, deterministic=True
)
eval_env.close()
print(f"mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")


- In my case, I got a mean reward of `200.20 +/- 20.80` after training for 1 million steps, which means that our lunar lander agent is ready to land on the moon 🌛🥳.

## Resume Training (optional) 🔄
If you want to **continue training from the last checkpoint** (e.g. with different hyperparameters), update the hyperparameters cell above and run the cell below.

The model weights, optimizer state and replay buffer are all restored — training picks up exactly where it left off.

In [ ]:
from stable_baselines3 import SAC
import os
import glob

from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.vec_env import VecNormalize

lr_callable = (
    cosine_lr_schedule(learning_rate_max, learning_rate_min)
    if lr_schedule.lower() == "cosine"
    else linear_lr_schedule(learning_rate_max, learning_rate_min)
)

latest = max(
    glob.glob(os.path.join(checkpoint_dir, f"{model_name}*.zip")),
    key=os.path.getmtime,
)
print(f"Loading checkpoint: {latest}")

venv = make_subproc_venv(n_envs, SEED)
env = VecNormalize.load(vecnormalize_path, venv)

model = SAC.load(
    latest,
    env=env,
    device=device,
    learning_rate=lr_callable,
    buffer_size=buffer_size,
    learning_starts=learning_starts,
    batch_size=batch_size,
    tau=tau,
    gamma=gamma,
    train_freq=train_freq,
    gradient_steps=gradient_steps,
    ent_coef="auto",
    target_entropy=target_entropy,
)

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(tensorboard_log, exist_ok=True)
_checkpoint_every = max(save_freq // n_envs, 1)
checkpoint_callback = CheckpointCallback(
    save_freq=_checkpoint_every,
    save_path=checkpoint_dir,
    name_prefix=model_name,
)
vecnormalize_cb = VecNormalizeSaveCallback(
    save_freq=_checkpoint_every,
    save_path=vecnormalize_path,
)
periodic_eval_cb = PeriodicEvalCallback(
    stats_path=vecnormalize_path,
    seed=SEED,
    eval_freq=periodic_eval_freq,
    n_eval_episodes=periodic_eval_episodes,
    env_id=env_id,
    verbose=1,
)
entropy_log_cb = EntropyLogCallback(log_freq=max(periodic_eval_freq, 1), verbose=1)
video_cb = VideoProgressCallback(
    eval_freq=video_eval_freq,
    vecnormalize_path=vecnormalize_path,
    seed=SEED,
)
live_plot_cb = LiveRewardPlotCallback(
    window=reward_plot_window, plot_freq=reward_plot_freq
)

model.learn(
    total_timesteps=total_timesteps,
    callback=[
        checkpoint_callback,
        vecnormalize_cb,
        periodic_eval_cb,
        entropy_log_cb,
        video_cb,
        live_plot_cb,
    ],
    reset_num_timesteps=False,
)

if isinstance(env, VecNormalize):
    env.save(vecnormalize_path)
model.save(os.path.join(checkpoint_dir, f"{model_name}-final"))


## Publish our trained model on the Hub 🔥
Now that we saw we got good results after the training, we can publish our trained model on the hub 🤗 with one line of code.

📚 The libraries documentation 👉 https://github.com/huggingface/huggingface_sb3/tree/main#hugging-face--x-stable-baselines3-v20

Here's an example of a Model Card (with Space Invaders):

By using `package_to_hub` **you evaluate, record a replay, generate a model card of your agent and push it to the hub**.

This way:
- You can **showcase our work** 🔥
- You can **visualize your agent playing** 👀
- You can **share with the community an agent that others can use** 💾
- You can **access a leaderboard 🏆 to see how well your agent is performing compared to your classmates** 👉 https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard


To be able to share your model with the community there are three more steps to follow:

1️⃣ (If it's not already done) create an account on Hugging Face ➡ https://huggingface.co/join

2️⃣ Sign in and then, you need to store your authentication token from the Hugging Face website.
- Create a new token (https://huggingface.co/settings/tokens) **with write role**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- Copy the token
- Run the cell below and paste the token

In [ ]:
notebook_login()
!git config --global credential.helper store

If you don't want to use a Google Colab or a Jupyter Notebook, you need to use this command instead: `huggingface-cli login`

3️⃣ We're now ready to push our trained agent to the 🤗 Hub 🔥 using `package_to_hub()` function

Let's fill the `package_to_hub` function:
- `model`: our trained model (**MultiInputPolicy**, dict obs: `state` + `pixels`).
- `model_name`: the name of the trained model that we defined in `model_save`
- `model_architecture`: the model architecture we used, in our case SAC
- `env_id`: the name of the environment, in our case `LunarLander-v3`
- `eval_env`: vectorized eval env with the **same wrappers and VecNormalize stats** as training
- `repo_id`: the name of the Hugging Face Hub Repository that will be created/updated `(repo_id = {username}/{repo_name})`

💡 **A good name is {username}/{model_architecture}-{env_id}**

- `commit_message`: message of the commit


In [ ]:
import inspect

from huggingface_sb3 import package_to_hub

## repo_id is the id of the model repository from the Hugging Face Hub (repo_id = {organization}/{repo_name} for instance username/sac-LunarLander-v3
repo_id = "ntitz19/sac-LunarLander-v3"

# TODO: Define the name of the environment
env_id = "LunarLander-v3"

eval_env = make_eval_vec_env_with_stats(vecnormalize_path, SEED)

# TODO: Define the model architecture we used
model_architecture = "SAC"

## TODO: Define the commit message
commit_message = "Upload SAC LunarLander-v3 MultiInput dict-obs agent"

_sig = inspect.signature(package_to_hub)
_kwargs = dict(
    model=model,
    model_name=model_name,
    model_architecture=model_architecture,
    env_id=env_id,
    eval_env=eval_env,
    repo_id=repo_id,
    commit_message=commit_message,
)
if "n_eval_episodes" in _sig.parameters:
    _kwargs["n_eval_episodes"] = 20
# If your huggingface_sb3 is older and lacks n_eval_episodes, the default episode count applies.

package_to_hub(**_kwargs)
eval_env.close()

# The course progress checker looks for "LunarLander-v2" tag, but the env is v3.
# Patch the model card to include the v2 tag so the checker recognizes it.
from huggingface_hub import ModelCard

card = ModelCard.load(repo_id)
if "LunarLander-v2" not in card.data.tags:
    card.data.tags.append("LunarLander-v2")
    card.push_to_hub(
        repo_id, commit_message="Add LunarLander-v2 tag for course certification compatibility"
    )


Congrats 🥳 you've just trained and uploaded your first Deep Reinforcement Learning agent. The script above should have displayed a link to a model repository such as https://huggingface.co/osanseviero/test_sb3. When you go to this link, you can:
* See a video preview of your agent at the right.
* Click "Files and versions" to see all the files in the repository.
* Click "Use in stable-baselines3" to get a code snippet that shows how to load the model.
* A model card (`README.md` file) which gives a description of the model

Under the hood, the Hub uses git-based repositories (don't worry if you don't know what git is), which means you can update the model with new versions as you experiment and improve your agent.

Compare the results of your Lunar Lander agent (this notebook uses **LunarLander-v3**) with your classmates using the leaderboard 🏆 👉 https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard

## Some additional challenges 🏆
The best way to learn **is to try things by your own**! As you saw, the current agent is not doing great. As a first suggestion, you can train for more steps. With 1,000,000 steps, we saw some great results!

In the [Leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) you will find your agents. Can you get to the top?

Here are some ideas to achieve so:
* Train more steps
* Try different hyperparameters for `SAC`. You can see them at https://stable-baselines3.readthedocs.io/en/master/modules/sac.html#parameters.
* Check the [Stable-Baselines3 documentation](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html) and try another model such as DQN.
* **Push your new trained model** on the Hub 🔥

**Compare the results of your LunarLander-v2 with your classmates** using the [leaderboard](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard) 🏆

Is moon landing too boring for you? Try to **change the environment**, why not use MountainCar-v0, CartPole-v1 or CarRacing-v0? Check how they work [using the gym documentation](https://www.gymlibrary.dev/) and have fun 🎉.

## Notebook setup (MultiInput + dict observations)

- This notebook trains **SAC** on **dict observations**: `state` (8-d vector) and **`pixels`** (grayscale **84×84**, **uint8**, shape `(1, 84, 84)`).
- **Image scaling** uses SB3 policy preprocessing (`normalize_images=True`); **do not** pre-divide pixels by 255 in the env.
- **VecNormalize** updates **only** the `state` key (`norm_obs_keys=["state"]`) and reward during training; **`pixels` are not** normalized by VecNormalize.
- **Manual evaluation**, **progress videos**, and **Hugging Face push** all load the same **`vecnormalize.pkl`** statistics as training (`training=False`, `norm_reward=False` on eval).
